<a href="https://colab.research.google.com/github/simichakravarty16/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026-10-16%20%E2%80%94%20Walmart%20Warm%20Up%20SKU%20Mess%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [46]:
import pandas as pd
PATH = '../content/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [47]:
df = cat.copy()

filtered = df[df['product_name'].str.lower().str.contains('pop')].copy()
print(f'Count: {len(filtered)}')
filtered


Count: 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [48]:
print(f'Strawberry Pop-Tart SKUs: {filtered['sku'].value_counts()}\n')
print(f'Strawberry Pop-Tart Names: {filtered['product_name'].value_counts()}\n')

strawberry = filtered[filtered['product_name'].str.lower().str.contains('strawberry')]

print(f'\nNumber of Pop-Tart rows: {len(filtered)}')
print(f'Number of Strawberry Pop-Tart rows: {len(strawberry)}')

Strawberry Pop-Tart SKUs: sku
PT-12            1
SKU#459812       1
POPTART-STRAW    1
PT_STRAWBERRY    1
Name: count, dtype: int64

Strawberry Pop-Tart Names: product_name
pop-tarts strawberry    2
Strawberry Pop-Tart     1
Pop-Tarts Strawberry    1
Name: count, dtype: int64


Number of Pop-Tart rows: 4
Number of Strawberry Pop-Tart rows: 4


**Pop-Tart rows that are a different product:** 0 rows

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [49]:
DICT = {
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'POPTART-STRAW': 'POPTART-STRAW',
    'PT_STRAWBERRY': 'POPTART-STRAW'
}

df['canonical_sku'] = df['sku'].map(DICT).fillna(df['sku'])
df.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size,canonical_sku
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack,POPTART-STRAW
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk,POPTART-STRAW
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack,POPTART-STRAW
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack,POPTART-STRAW
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack,WTR-24


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [50]:
print(f'Before:\n')
print(df["category"].value_counts(dropna=False))

df['normalized_category'] = df['category'].str.strip().str.title()

print(f'\nAfter:\n')
print(df["normalized_category"].value_counts(dropna=False))

# Even after normalization, there are still pairs with two labels for one thing:
# Breakfast and Breakfast Foods
# Grocery and Groceries
# Baby and Baby Products
# Alcohol and Alcoholic Beverages
# Emergency and Emergency Supplies

Before:

category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64

After:

normalized_category
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
Health                 2
Br

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [51]:
import re

def normalize(name):
  out = name.lower().strip()
  out = re.sub(r'[^\w\s]', '', out)
  return ' '.join(sorted(out.split()))

df['normalized_product_name'] = df['product_name'].apply(normalize)

df.groupby('normalized_product_name').filter(lambda x: x['sku'].nunique() > 1)

,sku,product_name,category,subcategory,unit_cost,supplier,package_size,canonical_sku,normalized_category,normalized_product_name
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack,POPTART-STRAW,Breakfast Foods,poptarts strawberry
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack,POPTART-STRAW,Breakfast,poptarts strawberry
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack,POPTART-STRAW,Breakfast,poptarts strawberry
8,BAT-AA4,AA Batteries 4-Pack,Emergency Supplies,AA,4.16,Kimberly-Clark,4pk,BAT-AA4,Emergency Supplies,4pack aa batteries
9,BATT_AA_4PK,AA Batteries 4-Pack,Emergency,AA,3.71,Anheuser-Busch,24 pack,BATT_AA_4PK,Emergency,4pack aa batteries
10,FLASH-01,FLASHLIGHT LED,Emergency,Flashlight,NaN,Frito-Lay,6-pack,FLASH-01,Emergency,flashlight led
11,LED-FLASH,FLASHLIGHT LED,EMERGENCY,NaN,8.20,Frito-Lay,single,LED-FLASH,Emergency,flashlight led
12,BEER-24,Light Beer (24-pack),Alcoholic Beverages,NaN,9.62,Hasbro,bundle,BEER-24,Alcoholic Beverages,24pack beer light
13,LTBEER24,Light Beer (24-pack),ALCOHOL,Light,7.61,Kellogg's,bundle,LTBEER24,Alcohol,24pack beer light
14,BG-MONO,Monopoly Board Game,ENTERTAINMENT,NaN,9.96,Kimberly-Clark,1pk,BG-MONO,Entertainment,board game monopoly


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [52]:
assert df['canonical_sku'].notna().all()
assert len(cat) == len(df)

print("All checks passed")

All checks passed


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

_your answer here_